# EMMa Checkpoint Evaluation

In [ ]:
from pathlib import Path

In [ ]:
CHECKPOINT = None  # e.g. Path("/path/to/CKPT")

if CHECKPOINT is None:
    raise ValueError("Please set `CHECKPOINT` to the path of the checkpoint to evaluate.")

CHECKPOINT = Path(CHECKPOINT)

if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {CHECKPOINT}")

In [ ]:
DEVICE = "cuda"
SPLITS = ("train", "val", "test")  # add "test" only for final locked evaluation
BATCH_SIZE = 64
WORKERS = 4
EVAL_SEED = 3621
N_MASS_BINS = 25
MIN_LEAF_CATEGORY_N = 30
MIN_PROFILE_N = 30
N_DENSITY_SPAN_BINS = 5
MIN_SINGLE_MATERIAL_PER_MATERIAL_N = 5
COUNTERFACTUAL_N = 64
COUNTERFACTUAL_MATERIALS = (
    "polypropylene",
    "nylon plastic",
    "cotton",
    "porcelain",
    "other aluminum",
    "cast iron",
    "carbide",
)
COUNTERFACTUAL_MATERIALS_N = len(COUNTERFACTUAL_MATERIALS)
COUNTERFACTUAL_MIN_PROTOTYPE_N = 20
COUNTERFACTUAL_MAX_PROTOTYPE_OFFBRANCH = 0.10
COUNTERFACTUAL_MAX_DENSITY_RELATIVE_SPAN = 0.25
POINTCLOUD_SCALE_N = 128
POINTCLOUD_SCALES = (0.50, 0.75, 1.00, 1.25, 1.50)
TEXT_ABLATION_N = 128

In [ ]:
# Exact leaf category keys: (category_id, depth).
GEOMETRY_CATEGORIES = {
    "mouse_pads": (3086, 4),
    "hardcover_executive_notebooks": (3106, 4),
    "composition_notebooks": (3109, 4),
    "marble_tiles": (1364, 5),
    "glass_tiles": (2921, 4),
    "ceramic_tiles": (2923, 4),
    "corrugated_boxes": (1486, 2),
}

FILL_LOW_CATEGORIES = {
    "watering_cans": (2655, 3),
    "sports_water_bottles": (1169, 3),
    "kitchen_trash_cans": (752, 3),
    "thermoses": (471, 3),
    "commuter_travel_mugs": (470, 3),
    "vases": (96, 2),
    "tumblers_water_glasses": (675, 4),
    "teapots_coffee_servers": (329, 5),
    "lunch_boxes": (806, 4),
    "planters": (2641, 3),
    "cookie_jars": (811, 4),
    "wastebaskets": (751, 3),
}
FILL_HIGH_CATEGORIES = {
    "cutting_boards": (446, 3),
    "combination_wrenches": (2621, 4),
    "dumbbells": (1565, 4),
    "marble_tiles": (1364, 5),
    "hardcover_executive_notebooks": (3106, 4),
}

PROFILE_CASE_STUDIES = {
    "watering_cans": (2655, 3),
    "sports_water_bottles": (1169, 3),
    "kitchen_trash_cans": (752, 3),
    "thermoses": (471, 3),
    "cutting_boards": (446, 3),
    "combination_wrenches": (2621, 4),
    "dumbbells": (1565, 4),
    "marble_tiles": (1364, 5),
    "wine_glasses": (315, 5),
    "coffee_cups_mugs": (293, 5),
    "mouse_pads": (3086, 4),
    "hardcover_executive_notebooks": (3106, 4),
    "portable_bluetooth_speakers": (1030, 3),
    "portable_power_banks": (2255, 3),
    "handheld_flashlights": (2074, 3),
}

## Setup


In [ ]:
import sys
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

In [ ]:
import copy
import hashlib
import json
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import spearmanr
from torch.utils.data import DataLoader, Dataset
from torch.utils.data._utils.collate import default_collate

from emma import EmmaDataset
from models.density_lut import load_density_lut
from training.checkpoints import load_model
from training.config import load_json
from training.data import build_dataset, model_inputs

In [ ]:
DATA_CONFIG = repo_root / "configs" / "data.json"
data_config = load_json(DATA_CONFIG)

EMMA_ROOT = Path(data_config["emma_root"])
ASINS = {name: Path(path) for name, path in data_config["asins"].items()}
SINGLE_MATERIAL_ASINS = Path(data_config["single_material_asins"])
SINGLE_MATERIAL_MATERIALS = Path(data_config["single_material_materials"])
DENSITY_LUT = Path(data_config["density_lut"])
DENSITY_RANGE_LUT = Path(data_config["density_range_lut"])

In [ ]:
REPORT_DIR = Path(repo_root / "artifacts"  / "checkpoint_evaluation") / (
    f"{CHECKPOINT.parent.name}_{CHECKPOINT.stem}"
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def save(df, name):
    df.to_csv(REPORT_DIR / name, index=False)
    return df

## Metrics Helpers


In [ ]:
LABEL_AWARE_METRICS = [
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "labelled_fraction",
    "mass_recall_at_1",
    "labelled_mass_fraction",
    "dominance_margin",
    "mass_dominance_margin",
]

COMPOSITION_ONLY_METRICS = [
    "max_fraction",
    "entropy",
    "effective_materials",
]

COMP_COLS = LABEL_AWARE_METRICS + COMPOSITION_ONLY_METRICS


def mass_metrics(target, pred):
    target = np.asarray(target, float)
    pred = np.asarray(pred, float)

    valid = np.isfinite(target) & np.isfinite(pred) & (target > 0) & (pred > 0)
    target, pred = target[valid], pred[valid]

    if len(target) == 0:
        return {"n": 0, "alde": np.nan}

    log_target = np.log(target)
    log_pred = np.log(pred)
    log_error = log_pred - log_target

    ratio = np.minimum(pred / target, target / pred)
    mse_log = np.mean(log_error**2)
    mean_log_error = np.mean(log_error)
    # corr = np.corrcoef(log_pred, log_target)[0, 1] if len(target) > 1 else np.nan

    if len(target) > 1 and np.std(log_pred) > 0 and np.std(log_target) > 0:
        corr = np.corrcoef(log_pred, log_target)[0, 1]
    else:
        corr = np.nan

    return {
        "n": len(target),
        "alde": np.mean(np.abs(log_error)),
        "ape": np.mean(np.abs(pred - target) / target),
        "mnre": np.mean(ratio),
        "rmse_log": np.sqrt(mse_log),
        "log10": np.mean(np.abs(np.log10(pred) - np.log10(target))),
        "silog": np.sqrt(max(mse_log - 0.5 * mean_log_error**2, 0)),
        "q": np.mean(ratio >= 0.5),
        "r_ls2": corr**2 if np.isfinite(corr) else np.nan,
        "signed_log_bias": mean_log_error,
        "log_ratio_std": np.std(log_error),
    }


def composition_metrics(comp, labels, densities):
    comp = np.asarray(comp, float)
    labels = np.asarray(labels, bool)
    densities = np.asarray(densities, float)

    n_samples, n_materials = comp.shape
    rows = np.arange(n_samples)

    raw_mass = comp * densities[None, :]
    mass_share = raw_mass / np.clip(raw_mass.sum(axis=1, keepdims=True), 1e-12, None)
    out = pd.DataFrame(index=rows)

    # Top-k composition recall
    for k in (1, 3, 5):
        top_k = min(k, n_materials)
        idx = np.argpartition(-comp, top_k - 1, axis=1)[:, :top_k]
        out[f"recall_at_{k}"] = np.take_along_axis(labels, idx, axis=1).any(axis=1).astype(float)

    # Largest labelled/unlabelled fractions
    labelled_max = np.where(labels, comp, -np.inf).max(axis=1)
    unlabelled_max = np.where(~labels, comp, -np.inf).max(axis=1)
    labelled_mass_max = np.where(labels, mass_share, -np.inf).max(axis=1)
    unlabelled_mass_max = np.where(~labels, mass_share, -np.inf).max(axis=1)

    unlabelled_max[~np.isfinite(unlabelled_max)] = 0
    unlabelled_mass_max[~np.isfinite(unlabelled_mass_max)] = 0

    # Label-aware metrics
    out["labelled_fraction"] = (comp * labels).sum(axis=1)
    out["mass_recall_at_1"] = labels[rows, raw_mass.argmax(axis=1)].astype(float)
    out["labelled_mass_fraction"] = (mass_share * labels).sum(axis=1)
    out["dominance_margin"] = labelled_max - unlabelled_max
    out["mass_dominance_margin"] = labelled_mass_max - unlabelled_mass_max

    # Label-aware metrics are undefined if there are no labels
    no_labels = ~labels.any(axis=1)
    out.loc[no_labels, LABEL_AWARE_METRICS] = np.nan

    # Label-independent metrics
    out["max_fraction"] = comp.max(axis=1)
    out["entropy"] = -(comp * np.log(np.clip(comp, 1e-8, None))).sum(axis=1)
    out["effective_materials"] = np.exp(out["entropy"])

    return out


def grouped_metrics(pred_df, mask, comp_metrics=None):
    mask = np.asarray(mask, bool)

    out = mass_metrics(
        pred_df.loc[mask, "target_mass"],
        pred_df.loc[mask, "predicted_mass"],
    )

    if comp_metrics is not None:
        out.update({col: comp_metrics.loc[mask, col].mean() for col in COMP_COLS})

    return out


def distribution_summary(df, metrics, label="metric"):
    quantiles = {"p10": 0.10, "p25": 0.25, "p50": 0.50, "p75": 0.75, "p90": 0.90}
    rows = []

    for metric in metrics:
        if metric not in df:
            continue

        x = df[metric].dropna().to_numpy(float)
        if len(x) == 0:
            continue

        rows.append({
            label: metric,
            "n_bins": len(x),
            "mean": x.mean(),
            "std": x.std(),
            **{name: np.quantile(x, q) for name, q in quantiles.items()},
            "min": x.min(),
            "max": x.max(),
        })

    return pd.DataFrame(rows)

## Profile Helpers 

In [ ]:
def profile_stats(x):
    x = np.asarray(x, float)

    centroid = x.mean(axis=0)
    l2 = np.sum((x - centroid) ** 2, axis=1)

    denom = np.linalg.norm(x, axis=1) * np.linalg.norm(centroid)
    cosine = np.divide(
        x @ centroid,
        denom,
        out=np.full(len(x), np.nan),
        where=denom > 0,
    )

    return centroid, {
        "l2_mean": l2.mean(),
        "cosine_mean": np.nanmean(cosine),
    }


def compare_profiles(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    diff = a - b
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    eps = 1e-12
    mid = 0.5 * (a + b)
    a_safe = np.clip(a, eps, None)
    b_safe = np.clip(b, eps, None)
    mid_safe = np.clip(mid, eps, None)

    js = 0.5 * np.sum(a * np.log(a_safe / mid_safe))
    js += 0.5 * np.sum(b * np.log(b_safe / mid_safe))

    return {
        "cosine": a @ b / denom if denom else np.nan,
        "l1": np.abs(diff).sum(),
        "l2": np.linalg.norm(diff),
        "js": js,
    }


def top_materials(vector, names, k=6):
    vector = np.asarray(vector)
    idx = np.argsort(-vector)[:k]
    return "; ".join(f"{names[i]}={vector[i]:.3f}" for i in idx)


def ranked_profile_text(vector, k=6):
    vector = np.asarray(vector)
    return "; ".join(f"rank{i + 1}={v:.3f}" for i, v in enumerate(vector[:k]))

## Material Hierarchy Helpers 

In [ ]:
def material_key(name):
    return " ".join(str(name).strip().lower().replace("_", " ").split())


def build_material_hierarchy(names, spec):
    names = list(names)
    keys = [material_key(name) for name in names]
    key_to_idx = {key: i for i, key in enumerate(keys)}

    parent = np.full(len(names), -1, dtype=int)
    children = {i: [] for i in range(len(names))}

    for child_name, parent_name in spec["parent"].items():
        child_idx = key_to_idx.get(material_key(child_name))
        parent_idx = None if parent_name is None else key_to_idx.get(material_key(parent_name))

        if child_idx is None or parent_idx is None:
            continue

        parent[child_idx] = parent_idx
        children[parent_idx].append(child_idx)

    ancestors = []
    depths = np.zeros(len(names), dtype=int)
    roots = np.zeros(len(names), dtype=int)

    for i in range(len(names)):
        chain = []
        current = i
        seen = set()

        while parent[current] >= 0 and current not in seen:
            seen.add(current)
            current = int(parent[current])
            chain.append(current)

        ancestors.append(tuple(chain))
        depths[i] = len(chain)
        roots[i] = current

    descendants = []
    for i in range(len(names)):
        stack = list(children[i])
        found = []

        while stack:
            child = stack.pop()
            found.append(child)
            stack.extend(children[child])

        descendants.append(tuple(found))

    return SimpleNamespace(
        names=names,
        keys=keys,
        key_to_idx=key_to_idx,
        parent=parent,
        children=children,
        ancestors=ancestors,
        descendants=descendants,
        depths=depths,
        roots=roots,
    )


def frontier_indices(label_row, hierarchy):
    positives = set(np.flatnonzero(np.asarray(label_row, bool)))

    return np.asarray([
        i for i in positives
        if not any(descendant in positives for descendant in hierarchy.descendants[i])
    ], dtype=int)


def hierarchy_relation(pred_idx, target_idx, hierarchy):
    if pred_idx == target_idx:
        return "exact"
    if pred_idx in hierarchy.ancestors[target_idx]:
        return "ancestor"
    if pred_idx in hierarchy.descendants[target_idx]:
        return "descendant"
    if hierarchy.roots[pred_idx] == hierarchy.roots[target_idx]:
        return "same_branch"
    return "unrelated"


def tree_distance(a, b, hierarchy):
    if hierarchy.roots[a] != hierarchy.roots[b]:
        return np.nan

    path_a = [a, *hierarchy.ancestors[a]]
    path_b = [b, *hierarchy.ancestors[b]]
    pos_b = {node: i for i, node in enumerate(path_b)}

    return min(
        i + pos_b[node]
        for i, node in enumerate(path_a)
        if node in pos_b
    )

## Filtering Helpers

In [ ]:
def category_mask(df, key):
    category_id, depth = key
    return (
        (df["leaf_category_id"].to_numpy(int) == category_id)
        & (df["leaf_category_depth"].to_numpy(int) == depth)
    )

## Load checkpoint and run inference


In [ ]:
device = torch.device(
    DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu"
)

model, checkpoint = load_model(CHECKPOINT)
model_config = checkpoint["model_config"]
experiment = {
    "data": checkpoint["data_selection"],
    "model": model_config,
}
model.to(device).eval()

In [ ]:
emma = EmmaDataset(EMMA_ROOT)

class DeterministicEvalDataset(Dataset):
    """Keep point-cloud downsampling repeatable across checkpoints."""
    def __init__(self, base, seed=0):
        self.base, self.seed = base, seed

    def __len__(self):
        return len(self.base)

    def __getattr__(self, name):
        return getattr(self.base, name)

    def __getitem__(self, i):
        asin = str(self.base.filenames_list[i])
        h = int.from_bytes(
            hashlib.blake2b(asin.encode(), digest_size=8).digest(), "little"
        ) & 0xffffffff
        np_state, py_state = np.random.get_state(), random.getstate()
        try:
            np.random.seed((self.seed + h) & 0xffffffff)
            random.seed((self.seed + h) & 0xffffffff)
            return self.base[i]
        finally:
            np.random.set_state(np_state)
            random.setstate(py_state)

def make_dataset(path):
    base = build_dataset(
        data_config,
        experiment,
        model_config,
        split="val",
        is_train=False,
        asins_path=str(path),
    )
    return DeterministicEvalDataset(base, EVAL_SEED)

datasets = {split: make_dataset(ASINS[split]) for split in SPLITS}

In [ ]:
material_names = emma.material_names
material_hierarchy = build_material_hierarchy(material_names, emma.material_hierarchy)

if model.material_densities is not None:
    material_densities = model.material_densities.detach().cpu().numpy().astype(float)
else:
    material_densities = np.asarray(
        load_density_lut(
            DENSITY_LUT,
            material_names,
            unit=data_config.get("density_lut_unit", "g_cm3"),
        ),
        float,
    )

In [ ]:
def clone_batch(batch):
    return {k: (v.clone() if torch.is_tensor(v) else copy.deepcopy(v)) for k, v in batch.items()}


def predict_batch(batch):
    with torch.inference_mode():
        pred = model(model_inputs(batch, model, device))
    
    if pred.composition is not None:
        comp = pred.composition.float().cpu().numpy()
        solid_density = comp @ material_densities
    else:
        comp = None
        solid_density = np.nan
        
    return {
        "mass": pred.mass.float().reshape(-1).cpu().numpy(),
        "density": pred.density.float().reshape(-1).cpu().numpy(),
        "volume": pred.volume.float().reshape(-1).cpu().numpy(),
        "fill": None if pred.fill_factor is None else pred.fill_factor.float().reshape(-1).cpu().numpy(),
        "composition": comp,
        "solid_mixture_density": np.asarray(solid_density).reshape(-1) if np.ndim(solid_density) else solid_density,
    }


def run_split(dataset):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    rows, compositions, targets = [], [], []

    with torch.inference_mode():
        for batch in loader:
            pred = model(model_inputs(batch, model, device, non_blocking=True))
            labels = batch["material_target"].bool().cpu().numpy()
            fill = np.full(len(labels), np.nan)
            if pred.fill_factor is not None:
                fill = pred.fill_factor.float().reshape(-1).cpu().numpy()

            for i, asin in enumerate(batch["filename"]):
                rec = emma.get_record(str(asin))
                raw_path = rec.get("category", [])
                valid_path = [
                    (int(cid), depth)
                    for depth, cid in enumerate(raw_path)
                    if int(cid) >= 0
                ]
                leaf_id, leaf_depth = valid_path[-1] if valid_path else (-1, -1)
                leaf_name = emma.cat_id_to_name.get((leaf_id, leaf_depth), "Unknown")

                dims = np.squeeze(
                    np.asarray(emma.get_package_dimensions(rec), dtype=float)
                ) * 0.0254
                box_volume = float(np.prod(dims)) if dims.shape == (3,) else np.nan

                rows.append({
                    "asin": str(asin),
                    "target_mass": float(batch["mass"].reshape(-1)[i]),
                    "predicted_mass": float(pred.mass.reshape(-1)[i].cpu()),
                    "predicted_density": float(pred.density.reshape(-1)[i].cpu()),
                    "predicted_volume": float(pred.volume.reshape(-1)[i].cpu()),
                    "fill_factor": float(fill[i]),
                    "package_box_volume": box_volume,
                    "leaf_category_id": leaf_id,
                    "leaf_category_depth": leaf_depth,
                    "leaf_category_name": str(leaf_name),
                })

            targets.append(labels)
            if pred.composition is not None:
                compositions.append(pred.composition.float().cpu().numpy())

    df = pd.DataFrame(rows)
    labels = np.concatenate(targets)
    comp = np.concatenate(compositions) if compositions else None
    df["log_error"] = np.log(df.predicted_mass) - np.log(df.target_mass)
    df["abs_log_error"] = np.abs(df.log_error)
    df["volume_box_ratio"] = df.predicted_volume / df.package_box_volume
    df["predicted_solid_volume"] = df.predicted_volume * df.fill_factor
    df["solid_volume_box_ratio"] = df.predicted_solid_volume / df.package_box_volume

    cm = None
    if comp is not None:
        cm = composition_metrics(comp, labels, material_densities)
        df = pd.concat([df.reset_index(drop=True), cm.add_prefix("comp_")], axis=1)

    return {
        "dataset": dataset,
        "pred": df,
        "targets": labels,
        "composition": comp,
        "comp": cm,
    }


results = {split: run_split(dataset) for split, dataset in datasets.items()}
print(
    checkpoint.get("paper_label") or checkpoint.get("experiment_name"),
    "|",
    {split: len(r["pred"]) for split, r in results.items()},
    "| geometry:",
    model.geometry_mode,
)

## 1. Overall Performance


In [ ]:
overall_rows = []
for split, r in results.items():
    mask = np.ones(len(r["pred"]), dtype=bool)
    overall_rows.append({"split": split, **grouped_metrics(r["pred"], mask, r["comp"])})

overall = save(pd.DataFrame(overall_rows), "overall.csv")
display(overall.round(4))

## 2. Mass-Binned Performance


In [ ]:
for split, r in results.items():
    df, cm = r["pred"], r["comp"]
    mass = df.target_mass.to_numpy(float)
    edges = np.geomspace(mass[mass > 0].min(), mass.max() * (1 + 1e-9), N_MASS_BINS + 1)
    bin_id = np.digitize(mass, edges[1:-1])

    rows = []
    for i in range(N_MASS_BINS):
        mask = bin_id == i
        rows.append({
            "split": split, "bin": i, "mass_min": edges[i], "mass_max": edges[i + 1],
            **grouped_metrics(df, mask, cm),
        })

    table = save(pd.DataFrame(rows), f"mass_bins_{split}.csv")
    print(split.upper())
    display(table.round(4))


## 3. Leaf-Category Performance


In [ ]:
CATEGORY_DISTRIBUTION_METRICS = [
    "alde", "ape", "q", "signed_log_bias",
    "recall_at_1", "labelled_fraction", "mass_recall_at_1",
    "labelled_mass_fraction", "dominance_margin", "entropy",
]

for split, r in results.items():
    df, cm = r["pred"], r["comp"]
    rows = []

    grouped = df.groupby(["leaf_category_id", "leaf_category_depth", "leaf_category_name"], dropna=False)
    for (cid, depth, name), idx in grouped.groups.items():
        mask = df.index.isin(idx)
        rows.append({
            "category_id": cid, "depth": depth, "category": name,
            **grouped_metrics(df, mask, cm),
        })

    category_metrics = pd.DataFrame(rows).sort_values("n", ascending=False)
    save(category_metrics, f"leaf_category_metrics_{split}.csv")

    eligible = category_metrics.query("n >= @MIN_LEAF_CATEGORY_N")
    summary = distribution_summary(eligible, CATEGORY_DISTRIBUTION_METRICS)
    summary.insert(0, "split", split)
    save(summary, f"leaf_category_metric_distribution_{split}.csv")

    print(f"{split.upper()} — {len(eligible)} leaf categories with n >= {MIN_LEAF_CATEGORY_N}")
    display(summary.round(4))


## 4. Curated Single-Material Study


In [ ]:
single_asins = np.load(SINGLE_MATERIAL_ASINS, allow_pickle=False).astype(str).reshape(-1)
single_materials = np.load(SINGLE_MATERIAL_MATERIALS, allow_pickle=False).astype(str).reshape(-1)
single_material_by_asin = dict(zip(single_asins, single_materials))

single_dataset = make_dataset(SINGLE_MATERIAL_ASINS)
single_result = run_split(single_dataset)
single_df = single_result["pred"].copy()
single_comp = single_result["composition"]
single_labels = single_result["targets"]

# Curated target + original EMMa target labels.
single_df["target_material"] = single_df.asin.map(single_material_by_asin)
single_df["target_material_index"] = single_df.target_material.map(
    lambda x: material_hierarchy.key_to_idx.get(material_key(x), -1)
)
single_df["emma_target_materials"] = [
    "; ".join(material_names[j] for j in np.where(row)[0]) for row in single_labels
]

In [ ]:
with open(DENSITY_RANGE_LUT) as f:
    density_ranges = {material_key(k): v for k, v in json.load(f).items()}

single_df["density_min"] = single_df.target_material.map(
    lambda x: density_ranges.get(material_key(x), {}).get("min", np.nan)
)
single_df["density_max"] = single_df.target_material.map(
    lambda x: density_ranges.get(material_key(x), {}).get("max", np.nan)
)
mid = (single_df.density_min + single_df.density_max) / 2
single_df["relative_span"] = (single_df.density_max - single_df.density_min) / mid

In [ ]:
if single_comp is not None:
    H = material_hierarchy
    raw_mass = single_comp * material_densities[None, :]
    mass_share = raw_mass / np.clip(raw_mass.sum(1, keepdims=True), 1e-12, None)
    top1 = single_comp.argmax(1)
    mass_top1 = raw_mass.argmax(1)

    hierarchy_rows = []
    for i, target_idx in enumerate(single_df.target_material_index.to_numpy(int)):
        if target_idx < 0:
            hierarchy_rows.append({})
            continue

        ancestors = set(H.ancestors[target_idx])
        descendants = set(H.descendants[target_idx])
        same_branch = {
            j for j in range(len(material_names))
            if H.roots[j] == H.roots[target_idx]
            and j != target_idx and j not in ancestors and j not in descendants
        }
        unrelated = set(range(len(material_names))) - {target_idx} - ancestors - descendants - same_branch

        groups = {
            "exact": {target_idx},
            "ancestor": ancestors,
            "descendant": descendants,
            "same_branch": same_branch,
            "unrelated": unrelated,
        }
        row = {
            "target_depth": int(H.depths[target_idx]),
            "target_root": material_names[H.roots[target_idx]],
            "target_path": " > ".join(reversed([material_names[target_idx], *[material_names[j] for j in H.ancestors[target_idx]]])),
            "pred_top1_material": material_names[top1[i]],
            "pred_top1_depth": int(H.depths[top1[i]]),
            "top1_relation": hierarchy_relation(int(top1[i]), target_idx, H),
            "top1_tree_distance": tree_distance(int(top1[i]), target_idx, H),
            "top1_depth_delta": int(H.depths[top1[i]] - H.depths[target_idx]),
            "mass_top1_material": material_names[mass_top1[i]],
            "mass_top1_relation": hierarchy_relation(int(mass_top1[i]), target_idx, H),
        }
        for relation, idxs in groups.items():
            idxs = list(idxs)
            row[f"{relation}_fraction"] = single_comp[i, idxs].sum() if idxs else 0.0
            row[f"{relation}_mass_fraction"] = mass_share[i, idxs].sum() if idxs else 0.0

        row["compatible_fraction"] = row["exact_fraction"] + row["ancestor_fraction"] + row["descendant_fraction"]
        row["compatible_mass_fraction"] = (
            row["exact_mass_fraction"] + row["ancestor_mass_fraction"] + row["descendant_mass_fraction"]
        )
        hierarchy_rows.append(row)

    hierarchy_df = pd.DataFrame(hierarchy_rows)
    single_df = pd.concat([single_df.reset_index(drop=True), hierarchy_df], axis=1)

summary = mass_metrics(single_df.target_mass, single_df.predicted_mass)
for col in (
    "exact_fraction", "ancestor_fraction", "descendant_fraction", "same_branch_fraction", "unrelated_fraction",
    "compatible_fraction", "exact_mass_fraction", "ancestor_mass_fraction", "descendant_mass_fraction",
    "same_branch_mass_fraction", "unrelated_mass_fraction", "compatible_mass_fraction",
):
    if col in single_df:
        summary[col] = single_df[col].mean()

if "top1_relation" in single_df:
    for relation in ("exact", "ancestor", "descendant", "same_branch", "unrelated"):
        summary[f"top1_{relation}_rate"] = (single_df.top1_relation == relation).mean()
    summary["top1_tree_distance_mean"] = single_df.top1_tree_distance.mean()
    summary["top1_depth_delta_mean"] = single_df.top1_depth_delta.mean()

summary["fill_factor_p50"] = single_df.fill_factor.median()
summary_df = save(pd.DataFrame([summary]), "single_material_summary.csv")
save(single_df, "single_material_objects.csv")
display(summary_df.round(4))

### 4a. Hierarchy-aware Specificity Diagnostics


In [ ]:
if single_comp is None:
    print("No composition output for this model.")
else:
    # How often is top-1 exact, merely coarser, more specific, same-branch, or unrelated?
    relation_rows = []
    for relation, group in single_df.groupby("top1_relation"):
        row = {"relation": relation, **mass_metrics(group.target_mass, group.predicted_mass)}
        for col in (
            "exact_fraction", "ancestor_fraction", "descendant_fraction", "same_branch_fraction",
            "unrelated_fraction", "compatible_fraction", "compatible_mass_fraction",
        ):
            row[col] = group[col].mean()
        row["tree_distance_mean"] = group.top1_tree_distance.mean()
        row["depth_delta_mean"] = group.top1_depth_delta.mean()
        relation_rows.append(row)
    relation_table = save(pd.DataFrame(relation_rows), "single_material_hierarchy_relations.csv")
    display(relation_table.round(4))

    # Does exact specificity become harder as the curated target gets deeper in the taxonomy?
    depth_rows = []
    for depth, group in single_df.groupby("target_depth"):
        depth_rows.append({
            "target_depth": depth,
            "n": len(group),
            "alde": group.abs_log_error.mean(),
            "exact_top1": (group.top1_relation == "exact").mean(),
            "ancestor_top1": (group.top1_relation == "ancestor").mean(),
            "descendant_top1": (group.top1_relation == "descendant").mean(),
            "same_branch_top1": (group.top1_relation == "same_branch").mean(),
            "unrelated_top1": (group.top1_relation == "unrelated").mean(),
            "exact_fraction": group.exact_fraction.mean(),
            "ancestor_fraction": group.ancestor_fraction.mean(),
            "compatible_fraction": group.compatible_fraction.mean(),
            "compatible_mass_fraction": group.compatible_mass_fraction.mean(),
            "top1_depth_delta_mean": group.top1_depth_delta.mean(),
        })
    depth_table = save(pd.DataFrame(depth_rows), "single_material_by_target_depth.csv")
    display(depth_table.round(4))

    # Per-target view: useful for spotting systematic ABS→plastic, cast iron→metal, etc.
    target_rows = []
    for material, group in single_df.groupby("target_material"):
        top = group.pred_top1_material.value_counts()
        target_rows.append({
            "target_material": material,
            "target_depth": group.target_depth.iloc[0],
            "n": len(group),
            "alde": group.abs_log_error.mean(),
            "exact_top1": (group.top1_relation == "exact").mean(),
            "ancestor_top1": (group.top1_relation == "ancestor").mean(),
            "same_branch_top1": (group.top1_relation == "same_branch").mean(),
            "unrelated_top1": (group.top1_relation == "unrelated").mean(),
            "exact_fraction": group.exact_fraction.mean(),
            "ancestor_fraction": group.ancestor_fraction.mean(),
            "compatible_fraction": group.compatible_fraction.mean(),
            "most_common_top1": top.index[0],
            "most_common_top1_rate": top.iloc[0] / len(group),
        })
    by_target = save(
        pd.DataFrame(target_rows).sort_values(["n", "ancestor_top1"], ascending=[False, False]),
        "single_material_by_target_material.csv",
    )
    display(by_target.head(30).round(4))


### 4b. Global Taxonomy Bias


In [ ]:
if results[next(iter(results))]["composition"] is None:
    print("No composition output for this model.")
else:
    H = material_hierarchy
    all_depth_rows, usage_rows = [], []

    for split, r in results.items():
        comp = r["composition"]
        labels = r["targets"]
        raw_mass = comp * material_densities[None, :]
        mass_share = raw_mass / np.clip(raw_mass.sum(1, keepdims=True), 1e-12, None)
        top1 = comp.argmax(1)

        for depth in sorted(set(H.depths)):
            idx = np.where(H.depths == depth)[0]
            all_depth_rows.append({
                "split": split,
                "depth": int(depth),
                "n_material_nodes": len(idx),
                "mean_composition_fraction": comp[:, idx].sum(1).mean(),
                "mean_mass_fraction": mass_share[:, idx].sum(1).mean(),
                "top1_rate": np.isin(top1, idx).mean(),
                "emma_label_share": labels[:, idx].sum() / max(labels.sum(), 1),
            })

        top1_counts = np.bincount(top1, minlength=len(material_names)) / len(comp)
        for j, name in enumerate(material_names):
            usage_rows.append({
                "split": split,
                "material": name,
                "depth": int(H.depths[j]),
                "root": material_names[H.roots[j]],
                "mean_fraction": comp[:, j].mean(),
                "mean_mass_fraction": mass_share[:, j].mean(),
                "top1_rate": top1_counts[j],
                "emma_label_rate": labels[:, j].mean(),
            })

    depth_usage = save(pd.DataFrame(all_depth_rows), "material_depth_usage.csv")
    material_usage = save(pd.DataFrame(usage_rows), "material_node_usage.csv")
    display(depth_usage.round(4))

    # Curated targets versus their predicted top-1 depths is the cleanest direct specificity comparison.
    target_depth_dist = (
        single_df.groupby(["target_depth", "pred_top1_depth"]).size().rename("n").reset_index()
    )
    save(target_depth_dist, "single_material_target_vs_predicted_depth.csv")

    split_to_show = "val" if "val" in results else next(iter(results))
    print(f"Most-used material nodes on {split_to_show}:")
    display(
        material_usage.query("split == @split_to_show")
        .sort_values("mean_fraction", ascending=False)
        .head(20).round(4)
    )

### 4c. Performance versus Material Density Relative Span


In [ ]:
# min == max gives relative_span == 0 and is intentionally excluded here.
span_df = single_df[np.isfinite(single_df.relative_span) & (single_df.relative_span > 0)].copy()

span_df["span_bin"] = pd.qcut(span_df.relative_span, q=N_DENSITY_SPAN_BINS, duplicates="drop")
rows = []
for interval, group in span_df.groupby("span_bin", observed=True):
    row = {
        "span_bin": str(interval),
        "span_min": group.relative_span.min(),
        "span_p50": group.relative_span.median(),
        "span_max": group.relative_span.max(),
        **mass_metrics(group.target_mass, group.predicted_mass),
    }
    for col in ("target_fraction", "target_mass_fraction", "target_top1", "target_mass_top1"):
        if col in group:
            row[col] = group[col].mean()
    rows.append(row)

span_bins = save(pd.DataFrame(rows), "single_material_density_span_bins.csv")
display(span_bins.round(4))

## 5. Volume / Geometry Diagnostics


In [ ]:
summary_rows = []
for split, r in results.items():
    df = r["pred"].copy()
    valid = df[np.isfinite(df.volume_box_ratio) & (df.volume_box_ratio > 0)]
    vs = valid.solid_volume_box_ratio.to_numpy(float)
    vs = vs[np.isfinite(vs) & (vs > 0)]

    rho_v = np.nan
    rho_vs = np.nan
    if len(valid) >= 3 and valid.package_box_volume.nunique() > 1:
        if valid.predicted_volume.nunique() > 1:
            rho_v = spearmanr(valid.package_box_volume, valid.predicted_volume).correlation
        valid_vs = valid[np.isfinite(valid.predicted_solid_volume)]
        if len(valid_vs) >= 3 and valid_vs.predicted_solid_volume.nunique() > 1:
            rho_vs = spearmanr(valid_vs.package_box_volume, valid_vs.predicted_solid_volume).correlation

    ratio = valid.volume_box_ratio.to_numpy(float)
    summary_rows.append({
        "split": split, "n": len(valid),
        "V_over_LWH_p10": np.quantile(ratio, 0.10) if len(ratio) else np.nan,
        "V_over_LWH_p50": np.median(ratio) if len(ratio) else np.nan,
        "V_over_LWH_p90": np.quantile(ratio, 0.90) if len(ratio) else np.nan,
        "Vs_over_LWH_p10": np.quantile(vs, 0.10) if len(vs) else np.nan,
        "Vs_over_LWH_p50": np.median(vs) if len(vs) else np.nan,
        "Vs_over_LWH_p90": np.quantile(vs, 0.90) if len(vs) else np.nan,
        "spearman_V_vs_LWH": rho_v,
        "spearman_Vs_vs_LWH": rho_vs,
    })

volume_summary = save(pd.DataFrame(summary_rows), "volume_scale_summary.csv")
display(volume_summary.round(4))

rows = []
for split, r in results.items():
    df = r["pred"]
    for label, key in GEOMETRY_CATEGORIES.items():
        group = df.loc[category_mask(df, key)].copy()
        group = group[np.isfinite(group.volume_box_ratio) & (group.volume_box_ratio > 0)]
        ratio = group.volume_box_ratio.to_numpy(float)
        solid_ratio = group.solid_volume_box_ratio.to_numpy(float)
        solid_ratio = solid_ratio[np.isfinite(solid_ratio) & (solid_ratio > 0)]

        rho_v = np.nan
        rho_vs = np.nan
        if len(group) >= 3 and group.package_box_volume.nunique() > 1:
            if group.predicted_volume.nunique() > 1:
                rho_v = spearmanr(group.package_box_volume, group.predicted_volume).correlation
            group_vs = group[np.isfinite(group.predicted_solid_volume)]
            if len(group_vs) >= 3 and group_vs.predicted_solid_volume.nunique() > 1:
                rho_vs = spearmanr(group_vs.package_box_volume, group_vs.predicted_solid_volume).correlation

        rows.append({
            "split": split, "case": label, "category_id": key[0], "depth": key[1], "n": len(group),
            "box_volume_p50": group.package_box_volume.median() if len(group) else np.nan,
            "predicted_volume_p50": group.predicted_volume.median() if len(group) else np.nan,
            "predicted_solid_volume_p50": group.predicted_solid_volume.median() if len(group) else np.nan,
            "V_over_LWH_p10": np.quantile(ratio, 0.10) if len(ratio) else np.nan,
            "V_over_LWH_p50": np.median(ratio) if len(ratio) else np.nan,
            "V_over_LWH_p90": np.quantile(ratio, 0.90) if len(ratio) else np.nan,
            "Vs_over_LWH_p10": np.quantile(solid_ratio, 0.10) if len(solid_ratio) else np.nan,
            "Vs_over_LWH_p50": np.median(solid_ratio) if len(solid_ratio) else np.nan,
            "Vs_over_LWH_p90": np.quantile(solid_ratio, 0.90) if len(solid_ratio) else np.nan,
            "spearman_V_vs_LWH": rho_v,
            "spearman_Vs_vs_LWH": rho_vs,
        })

geometry = save(pd.DataFrame(rows), "geometry_diagnostics.csv")
display(geometry.round(4))

### 5a. Point-Cloud Scale Equivariance


In [ ]:
split = "val" if "val" in results else next(iter(results))
ds = results[split]["dataset"]
rng = np.random.default_rng(EVAL_SEED)
chosen = rng.choice(len(ds), min(POINTCLOUD_SCALE_N, len(ds)), replace=False)

rows = []
composition_cache = {}
for i in chosen:
    base = default_collate([ds[int(i)]])
    asin = str(base["filename"][0])

    for scale in POINTCLOUD_SCALES:
        batch = clone_batch(base)
        batch["pc_incomplete"] = batch["pc_incomplete"] * float(scale)
        out = predict_batch(batch)
        comp = None if out["composition"] is None else out["composition"][0]
        composition_cache[(asin, float(scale))] = comp

        rows.append({
            "asin": asin, "scale": float(scale),
            "mass": float(out["mass"][0]),
            "density": float(out["density"][0]),
            "volume": float(out["volume"][0]),
            "fill": np.nan if out["fill"] is None else float(out["fill"][0]),
            "solid_mixture_density": np.nan if out["composition"] is None else float(out["solid_mixture_density"][0]),
            "composition_l1_from_scale1": np.nan,
        })

# Add ratios relative to each object's k=1 prediction.
scale_df = pd.DataFrame(rows)
for asin, idx in scale_df.groupby("asin").groups.items():
    idx = list(idx)
    group = scale_df.loc[idx]
    ref = group.loc[np.isclose(group.scale, 1.0)].iloc[0]
    scale_df.loc[idx, "volume_ratio"] = group.volume / ref.volume
    scale_df.loc[idx, "mass_ratio"] = group.mass / ref.mass
    scale_df.loc[idx, "density_ratio"] = group.density / ref.density
    if np.isfinite(ref.fill):
        scale_df.loc[idx, "fill_ratio"] = group.fill / ref.fill

    ref_comp = composition_cache.get((asin, 1.0))
    if ref_comp is not None:
        for scale in POINTCLOUD_SCALES:
            comp = composition_cache[(asin, float(scale))]
            mask = (scale_df.asin == asin) & np.isclose(scale_df.scale, scale)
            scale_df.loc[mask, "composition_l1_from_scale1"] = np.abs(comp - ref_comp).sum()

scale_df["expected_volume_ratio"] = scale_df.scale ** 3
scale_df["volume_equivariance_log_error"] = np.abs(
    np.log(np.clip(scale_df.volume_ratio, 1e-12, None)) - 3 * np.log(scale_df.scale)
)
scale_df["mass_equivariance_log_error"] = np.abs(
    np.log(np.clip(scale_df.mass_ratio, 1e-12, None)) - 3 * np.log(scale_df.scale)
)
save(scale_df, "pointcloud_scaling.csv")

In [ ]:
summary_rows = []
for scale, group in scale_df.groupby("scale"):
    summary_rows.append({
        "scale": scale,
        "expected_ratio": scale ** 3,
        "volume_ratio_p50": group.volume_ratio.median(),
        "volume_ratio_p10": group.volume_ratio.quantile(0.10),
        "volume_ratio_p90": group.volume_ratio.quantile(0.90),
        "mass_ratio_p50": group.mass_ratio.median(),
        "density_ratio_p50": group.density_ratio.median(),
        "fill_ratio_p50": group.fill_ratio.median() if "fill_ratio" in group else np.nan,
        "composition_l1_p50": group.composition_l1_from_scale1.median(),
        "volume_equivariance_log_error_p50": group.volume_equivariance_log_error.median(),
    })
scale_summary = save(pd.DataFrame(summary_rows), "pointcloud_scaling_summary.csv")
display(scale_summary.round(4))

object_rows = []
for asin, group in scale_df.groupby("asin"):
    group = group.sort_values("scale")
    x = np.log(group.scale.to_numpy(float))
    volume_slope = np.polyfit(x, np.log(group.volume.to_numpy(float)), 1)[0]
    mass_slope = np.polyfit(x, np.log(group.mass.to_numpy(float)), 1)[0]
    if (len(group.scale.to_numpy(float)) > 1 and np.unique(group.scale.to_numpy(float)).size > 1 and np.unique(group.volume.to_numpy(float)).size > 1):
        rho = spearmanr(group.scale.to_numpy(float), group.volume.to_numpy(float)).correlation
    else:
        rho = np.nan
    object_rows.append({
        "asin": asin,
        "volume_loglog_slope": volume_slope,
        "mass_loglog_slope": mass_slope,
        "spearman_scale_vs_volume": rho,
        "volume_equivariance_log_mae": group.volume_equivariance_log_error.mean(),
        "mass_equivariance_log_mae": group.mass_equivariance_log_error.mean(),
        "density_relative_range": (group.density.max() - group.density.min()) / max(abs(group.density.mean()), 1e-12),
        "fill_range": group.fill.max() - group.fill.min() if np.isfinite(group.fill).any() else np.nan,
        "composition_l1_max": group.composition_l1_from_scale1.max(),
    })
scale_objects = save(pd.DataFrame(object_rows), "pointcloud_scaling_by_object.csv")
display(pd.DataFrame([{
    "n": len(scale_objects),
    "volume_slope_p50": scale_objects.volume_loglog_slope.median(),
    "volume_slope_p10": scale_objects.volume_loglog_slope.quantile(0.10),
    "volume_slope_p90": scale_objects.volume_loglog_slope.quantile(0.90),
    "mass_slope_p50": scale_objects.mass_loglog_slope.median(),
    "volume_equivariance_log_mae_p50": scale_objects.volume_equivariance_log_mae.median(),
    "density_relative_range_p50": scale_objects.density_relative_range.median(),
    "fill_range_p50": scale_objects.fill_range.median(),
    "composition_l1_max_p50": scale_objects.composition_l1_max.median(),
}]).round(4))

## 6. Fill-Factor Analysis


In [ ]:
if not model.uses_fill_factor:
    print("No fill-factor output for this model.")
else:
    category_rows, group_rows = [], []
    groups = {"low": FILL_LOW_CATEGORIES, "high": FILL_HIGH_CATEGORIES}

    for split, r in results.items():
        df = r["pred"]
        for expected, categories in groups.items():
            pooled = []
            for label, key in categories.items():
                values = df.loc[category_mask(df, key), "fill_factor"].dropna().to_numpy(float)
                pooled.extend(values.tolist())
                category_rows.append({
                    "split": split, "expected_fill": expected, "case": label,
                    "category_id": key[0], "depth": key[1], "n": len(values),
                    "fill_mean": np.mean(values) if len(values) else np.nan,
                    "fill_p10": np.quantile(values, 0.10) if len(values) else np.nan,
                    "fill_p50": np.median(values) if len(values) else np.nan,
                    "fill_p90": np.quantile(values, 0.90) if len(values) else np.nan,
                })

            pooled = np.asarray(pooled, float)
            group_rows.append({
                "split": split, "expected_fill": expected, "n": len(pooled),
                "fill_mean": np.mean(pooled) if len(pooled) else np.nan,
                "fill_p10": np.quantile(pooled, 0.10) if len(pooled) else np.nan,
                "fill_p50": np.median(pooled) if len(pooled) else np.nan,
                "fill_p90": np.quantile(pooled, 0.90) if len(pooled) else np.nan,
            })

    fill_categories = save(pd.DataFrame(category_rows), "fill_factor_categories.csv")
    fill_groups = save(pd.DataFrame(group_rows), "fill_factor_groups.csv")
    display(fill_groups.round(4))
    display(fill_categories.round(4))

## 7. Leaf-Category Composition Profiles


In [ ]:
if results[next(iter(results))]["composition"] is None:
    print("No composition output for this model.")
else:
    profile_rows = []
    centroids = {}

    for split, r in results.items():
        df, comp, cm = r["pred"], r["composition"], r["comp"]
        grouped = df.groupby(["leaf_category_id", "leaf_category_depth", "leaf_category_name"], dropna=False)

        for (cid, depth, name), idx in grouped.groups.items():
            idx = np.asarray(list(idx), int)
            full_centroid, full_stats = profile_stats(comp[idx])
            ranked = np.sort(comp[idx], axis=1)[:, ::-1]
            ranked_centroid, ranked_stats = profile_stats(ranked)
            key = (int(cid), int(depth))
            centroids[(split, key)] = {
                "n": len(idx), "name": name,
                "full": full_centroid, "ranked": ranked_centroid,
            }

            profile_rows.append({
                "split": split, "category_id": cid, "depth": depth, "category": name, "n": len(idx),
                "full_l2_mean": full_stats["l2_mean"], "full_cosine_mean": full_stats["cosine_mean"],
                "ranked_l2_mean": ranked_stats["l2_mean"], "ranked_cosine_mean": ranked_stats["cosine_mean"],
                "labelled_fraction": cm.loc[idx, "labelled_fraction"].mean(),
                "labelled_mass_fraction": cm.loc[idx, "labelled_mass_fraction"].mean(),
                "fill_p50": df.loc[idx, "fill_factor"].median(),
                "full_profile": top_materials(full_centroid, material_names),
                "proportion_profile": ranked_profile_text(ranked_centroid),
            })

    profile_df = save(pd.DataFrame(profile_rows), "leaf_category_profiles.csv")

### 7a. Train↔Validation Profile Consistency


In [ ]:
if results[next(iter(results))]["composition"] is not None and "train" in results and "val" in results:
    rows = []
    train_keys = {key for split, key in centroids if split == "train"}
    val_keys = {key for split, key in centroids if split == "val"}

    for key in sorted(train_keys & val_keys):
        a, b = centroids[("train", key)], centroids[("val", key)]
        if a["n"] < MIN_PROFILE_N or b["n"] < MIN_PROFILE_N:
            continue
        full = compare_profiles(a["full"], b["full"])
        ranked = compare_profiles(a["ranked"], b["ranked"])
        rows.append({
            "category_id": key[0], "depth": key[1], "category": a["name"],
            "n_train": a["n"], "n_val": b["n"],
            **{f"full_{k}": v for k, v in full.items()},
            **{f"proportion_{k}": v for k, v in ranked.items()},
        })

    train_val_profiles = save(pd.DataFrame(rows), "leaf_profile_train_val_consistency.csv")
    summary = distribution_summary(
        train_val_profiles,
        ["full_cosine", "full_js", "proportion_cosine", "proportion_js"],
    )
    save(summary, "leaf_profile_train_val_consistency_distribution.csv")
    display(summary.round(4))

### 7b. Pairwise Similarity Between Leaf-Category Centroids


In [ ]:
def pairwise_centroid_similarity(split, profile_type):
    entries = [
        (key, value) for (s, key), value in centroids.items()
        if s == split and value["n"] >= MIN_PROFILE_N
    ]
    if len(entries) < 2:
        return None, None

    keys = [key for key, _ in entries]
    names = [value["name"] for _, value in entries]
    matrix = np.stack([value[profile_type] for _, value in entries])
    matrix = matrix / np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)
    sim = matrix @ matrix.T
    i, j = np.triu_indices(len(entries), 1)
    values = sim[i, j]

    summary = {
        "split": split, "profile_type": profile_type, "n_categories": len(entries), "n_pairs": len(values),
        "mean": values.mean(), "p10": np.quantile(values, 0.10), "p50": np.median(values),
        "p90": np.quantile(values, 0.90), "min": values.min(), "max": values.max(),
    }

    top = np.argsort(-values)[:20]
    pairs = pd.DataFrame({
        "split": split, "profile_type": profile_type,
        "category_a": [names[i[t]] for t in top],
        "category_b": [names[j[t]] for t in top],
        "cosine": values[top],
    })
    return summary, pairs


if results[next(iter(results))]["composition"] is not None:
    summaries, pair_tables = [], []
    for split in results:
        for profile_type in ("full", "ranked"):
            summary, pairs = pairwise_centroid_similarity(split, profile_type)
            if summary is not None:
                summaries.append(summary)
                pair_tables.append(pairs)

    pairwise_summary = save(pd.DataFrame(summaries), "leaf_profile_pairwise_similarity.csv")
    if pair_tables:
        save(pd.concat(pair_tables, ignore_index=True), "leaf_profile_most_similar_pairs.csv")
    display(pairwise_summary.round(4))

## 8. Profile Case Studies


In [ ]:
if results[next(iter(results))]["composition"] is None:
    print("No composition output for this model.")
else:
    rows = []
    case_centroids = {}

    for label, key in PROFILE_CASE_STUDIES.items():
        for split, r in results.items():
            df, comp, cm = r["pred"], r["composition"], r["comp"]
            mask = category_mask(df, key)
            idx = np.where(mask)[0]
            row = {"case": label, "split": split, "category_id": key[0], "depth": key[1], "n": len(idx)}

            if len(idx):
                full_centroid, full_stats = profile_stats(comp[idx])
                ranked = np.sort(comp[idx], axis=1)[:, ::-1]
                ranked_centroid, ranked_stats = profile_stats(ranked)
                case_centroids[(split, label)] = (full_centroid, ranked_centroid)

                core = grouped_metrics(df, mask, cm)
                row.update({k: core.get(k) for k in (
                    "alde", "q", "recall_at_1", "labelled_fraction",
                    "mass_recall_at_1", "labelled_mass_fraction",
                )})
                row.update({
                    "fill_p50": df.loc[mask, "fill_factor"].median(),
                    "full_cosine_mean": full_stats["cosine_mean"],
                    "proportion_cosine_mean": ranked_stats["cosine_mean"],
                    "full_profile": top_materials(full_centroid, material_names),
                    "proportion_profile": ranked_profile_text(ranked_centroid),
                })
            rows.append(row)

    case_studies = save(pd.DataFrame(rows), "profile_case_studies.csv")
    display(case_studies.round(4))

    if "train" in results and "val" in results:
        rows = []
        for label in PROFILE_CASE_STUDIES:
            if ("train", label) not in case_centroids or ("val", label) not in case_centroids:
                continue
            train_full, train_ranked = case_centroids[("train", label)]
            val_full, val_ranked = case_centroids[("val", label)]
            full = compare_profiles(train_full, val_full)
            ranked = compare_profiles(train_ranked, val_ranked)
            rows.append({
                "case": label,
                **{f"full_{k}": v for k, v in full.items()},
                **{f"proportion_{k}": v for k, v in ranked.items()},
            })
        display(save(pd.DataFrame(rows), "profile_case_train_val_consistency.csv").round(4))

## 9. Single-Material Counterfactuals


In [ ]:
def pairwise_order_accuracy(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    i, j = np.triu_indices(len(x), 1)
    keep = x[i] != x[j]
    if not keep.any():
        return np.nan
    return np.mean((x[i][keep] - x[j][keep]) * (y[i][keep] - y[j][keep]) > 0)

In [ ]:
if not model.uses_material:
    print("B1 has no material-prior input.")

if model.uses_material:
    H = material_hierarchy

    train_asins = set(results["train"]["pred"].asin.astype(str))
    val_asins = set(results["val"]["pred"].asin.astype(str))

    prior_by_asin = {}
    meta_rows = []
    for asin in single_df.asin.astype(str):
        material = single_material_by_asin[asin]
        j = material_hierarchy.key_to_idx.get(material_key(material), -1)
        if j < 0:
            continue

        rec = emma.get_record(asin)
        prior = np.asarray(emma.get_material_probs(rec), dtype=np.float32).reshape(-1)[:len(material_names)]
        prior_by_asin[asin] = prior
        meta_rows.append({
            "asin": asin,
            "frontier_index": j,
            "frontier_material": material_names[j],
            "is_train": asin in train_asins,
            "is_val": asin in val_asins,
        })

    cf_meta = pd.DataFrame(meta_rows)

    prototypes = {}
    prototype_rows = []
    donors = cf_meta.query("is_train")
    for j, group in donors.groupby("frontier_index"):
        vectors = np.stack([prior_by_asin[a] for a in group.asin])
        if len(vectors) < COUNTERFACTUAL_MIN_PROTOTYPE_N:
            continue
        j = int(j)
        proto = np.median(vectors, axis=0).astype(np.float32)
        prototypes[j] = proto
        density_info = density_ranges.get(material_key(material_names[j]), {})
        low = density_info.get("min", np.nan)
        high = density_info.get("max", np.nan)
        mid = (low + high) / 2 if np.isfinite(low) and np.isfinite(high) else np.nan
        span = (high - low) / mid if np.isfinite(mid) and mid > 0 else np.nan
        allowed = {j, *H.ancestors[j]}
        offbranch = [k for k in range(len(material_names)) if k not in allowed]
        prototype_rows.append({
            "material_index": j,
            "material": material_names[j],
            "n_train_donors": len(vectors),
            "lut_density": material_densities[j],
            "density_relative_span": span,
            "prototype_target_prior": proto[j],
            "prototype_offbranch_max": proto[offbranch].max() if offbranch else 0.0,
        })

    prototype_table = pd.DataFrame(prototype_rows)
    prototype_table = prototype_table[np.isfinite(prototype_table.lut_density) & (prototype_table.lut_density > 0)]
    prototype_table = prototype_table[prototype_table.prototype_offbranch_max <= COUNTERFACTUAL_MAX_PROTOTYPE_OFFBRANCH]
    prototype_table = prototype_table[prototype_table.density_relative_span <= COUNTERFACTUAL_MAX_DENSITY_RELATIVE_SPAN]

    selected_prototypes = (
        prototype_table[
            prototype_table.material.isin(COUNTERFACTUAL_MATERIALS)
        ]
        .sort_values("lut_density")
        .reset_index(drop=True)
    )

    missing_materials = set(COUNTERFACTUAL_MATERIALS) - set(selected_prototypes.material)
    if missing_materials:
        raise ValueError(
            f"Missing fixed counterfactual materials: {sorted(missing_materials)}"
        )

    save(selected_prototypes, "counterfactual_material_prototypes.csv")
    display(selected_prototypes.round(4))

    sources = cf_meta.query("is_val").copy()
    rng = np.random.default_rng(EVAL_SEED)
    if len(sources) > COUNTERFACTUAL_N:
        sources = sources.iloc[rng.choice(len(sources), COUNTERFACTUAL_N, replace=False)]

In [ ]:
if model.uses_material:
    single_index = {str(asin): i for i, asin in enumerate(single_df.asin)}
    rows = []
    for _, source in sources.iterrows():
        asin = source.asin
        base = default_collate([single_dataset[single_index[asin]]])
        original = predict_batch(base)
        original_comp = None if original["composition"] is None else original["composition"][0]

        for _, proto_row in selected_prototypes.iterrows():
            j = int(proto_row.material_index)
            batch = clone_batch(base)
            batch["material_prior"] = torch.as_tensor(prototypes[j], dtype=batch["material_prior"].dtype).unsqueeze(0)
            out = predict_batch(batch)
            comp = None if out["composition"] is None else out["composition"][0]
            compatible = [j, *H.ancestors[j], *H.descendants[j]]

            rows.append({
                "asin": asin,
                "source_material": source.frontier_material,
                "intervention_material": material_names[j],
                "intervention_density": material_densities[j],
                "intervention_density_relative_span": proto_row.density_relative_span,
                "mass": float(out["mass"][0]),
                "effective_density": float(out["density"][0]),
                "solid_mixture_density": np.nan if comp is None else float(out["solid_mixture_density"][0]),
                "volume": float(out["volume"][0]),
                "fill": np.nan if out["fill"] is None else float(out["fill"][0]),
                "intervention_exact_fraction": np.nan if comp is None else float(comp[j]),
                "intervention_compatible_fraction": np.nan if comp is None else float(comp[compatible].sum()),
                "composition_l1_from_original": np.nan if comp is None else float(np.abs(comp - original_comp).sum()),
            })

    counterfactuals = save(pd.DataFrame(rows), "counterfactuals.csv")

In [ ]:
if model.uses_material:
    summary_rows = []
    for asin, group in counterfactuals.groupby("asin"):
        group = group.sort_values("intervention_density")
        x = group.intervention_density.to_numpy(float)
        summary_rows.append({
            "asin": asin,
            "source_material": group.source_material.iloc[0],
            "mass_spearman": spearmanr(x, group.mass).correlation,
            "effective_density_spearman": spearmanr(x, group.effective_density).correlation,
            "solid_mixture_density_spearman": spearmanr(x, group.solid_mixture_density).correlation,
            "mass_pairwise_order_accuracy": pairwise_order_accuracy(x, group.mass),
            "effective_density_pairwise_order_accuracy": pairwise_order_accuracy(x, group.effective_density),
            "solid_density_pairwise_order_accuracy": pairwise_order_accuracy(x, group.solid_mixture_density),
            "volume_relative_range": (group.volume.max() - group.volume.min()) / max(abs(group.volume.mean()), 1e-12),
            "fill_range": group.fill.max() - group.fill.min() if np.isfinite(group.fill).any() else np.nan,
            "intervention_exact_fraction_mean": group.intervention_exact_fraction.mean(),
            "intervention_compatible_fraction_mean": group.intervention_compatible_fraction.mean(),
            "composition_l1_mean": group.composition_l1_from_original.mean(),
        })

    cf_by_object = save(pd.DataFrame(summary_rows), "counterfactual_summary_by_object.csv")
    cf_summary = pd.DataFrame([{
        "n_source_objects": len(cf_by_object),
        "n_intervention_materials": len(selected_prototypes),
        "mass_spearman_p50": cf_by_object.mass_spearman.median(),
        "effective_density_spearman_p50": cf_by_object.effective_density_spearman.median(),
        "solid_mixture_density_spearman_p50": cf_by_object.solid_mixture_density_spearman.median(),
        "mass_order_accuracy_mean": cf_by_object.mass_pairwise_order_accuracy.mean(),
        "effective_density_order_accuracy_mean": cf_by_object.effective_density_pairwise_order_accuracy.mean(),
        "solid_density_order_accuracy_mean": cf_by_object.solid_density_pairwise_order_accuracy.mean(),
        "intervention_exact_fraction_mean": counterfactuals.intervention_exact_fraction.mean(),
        "intervention_compatible_fraction_mean": counterfactuals.intervention_compatible_fraction.mean(),
        "max_volume_relative_range": cf_by_object.volume_relative_range.max(),
        "max_fill_range": cf_by_object.fill_range.max(),
    }])
    save(cf_summary, "counterfactual_summary.csv")
    display(cf_summary.round(4))

## 10. Text Ablation


In [ ]:
if not model.uses_text:
    print("This model is not text-conditioned.")
else:
    split = "val" if "val" in results else next(iter(results))
    ds = results[split]["dataset"]
    rng = np.random.default_rng(0)
    chosen = rng.choice(len(ds), min(TEXT_ABLATION_N, len(ds)), replace=False)
    donors = rng.permutation(chosen)

    rows = []
    for i, j in zip(chosen, donors):
        base = default_collate([ds[int(i)]])
        original = predict_batch(base)

        zero = clone_batch(base)
        zero["text_embedding"] = torch.zeros_like(zero["text_embedding"])

        shuffled = clone_batch(base)
        donor = ds[int(j)]["text_embedding"]
        shuffled["text_embedding"] = torch.as_tensor(donor).unsqueeze(0)

        for name, batch in (("original", base), ("zero", zero), ("shuffled", shuffled)):
            out = original if name == "original" else predict_batch(batch)
            row = {
                "mass": float(out["mass"][0]),
                "density": float(out["density"][0]),
                "volume": float(out["volume"][0]),
                "fill": np.nan if out["fill"] is None else float(out["fill"][0]),
            }
            row["intervention"] = name
            if out["composition"] is not None:
                row["composition_l1"] = 0 if name == "original" else np.abs(out["composition"][0] - original["composition"][0]).sum()
                row["top1_changed"] = False if name == "original" else out["composition"][0].argmax() != original["composition"][0].argmax()
            rows.append(row)

    text_df = save(pd.DataFrame(rows), "text_ablation.csv")
    baseline = text_df.query("intervention == 'original'").reset_index(drop=True)
    summary = []

    for name in ("zero", "shuffled"):
        altered = text_df.query("intervention == @name").reset_index(drop=True)
        rel = lambda col: np.median(np.abs(altered[col] - baseline[col]) / np.clip(np.abs(baseline[col]), 1e-12, None))
        row = {
            "intervention": name,
            "mass_change": rel("mass"), "density_change": rel("density"), "volume_change": rel("volume"),
        }
        if "composition_l1" in altered:
            row["composition_l1"] = altered.composition_l1.median()
            row["top1_changed"] = altered.top1_changed.mean()
        summary.append(row)

    display(pd.DataFrame(summary).round(5))